In [2]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import AIMessage,HumanMessage

In [3]:
load_dotenv()

True

In [6]:
llm=ChatGroq(model="llama-3.1-8b-instant")
parser=StrOutputParser()
prompt=ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant.\n\n"
     "Summary of old conversation:\n{summary}"),
    MessagesPlaceholder(variable_name="recent_messages"),
    ("human", "{input}")])

In [20]:
chain=prompt|llm|parser
# ── Summarizer chain ──────────────────────────────────────
summary_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Summarize this conversation in 2 sentences. "
     "Keep all names, goals, locations, key facts."),
    ("human", "{conversation}")
])
summary_chain=summary_prompt|llm|parser
all_messages={}
summaries={}
MAX_RECENT=4
def chat(message,session_id="s1"):
    messages=all_messages.get(session_id,[])
    summary=summaries.get(session_id,"no conversation yet")
    recent=messages[-MAX_RECENT:] if len(messages)>MAX_RECENT else messages
    response=chain.invoke({
    "input":message,
    "summary":summary,
    "recent_messages":recent})
    print(f"You: {message}")
    print(f"AI : {response}")
    messages.append(HumanMessage(content=response))
    messages.append(AIMessage(content=response))
    all_messages[session_id]=messages
    if len(messages) > MAX_RECENT:
            old      = messages[:-MAX_RECENT]
            old_text = ""
            for m in old:
                role      = "Human" if isinstance(m, HumanMessage) else "AI"
                old_text += f"{role}: {m.content}\n"
    
            summaries[session_id] = summary_chain.invoke({
                "conversation": f"Previous summary: {summary}\n\nOld messages:\n{old_text}"
            })
            summary = summaries[session_id]
    
    print(f"Total messages : {len(messages)}")
    print(f"Recent in LLM  : {min(len(messages), MAX_RECENT)} raw messages")
    print(f"Summary        : {summary[:80]}...")
    print()

In [22]:
chat("My name is Krish.")
chat("I am from Ahmedabad.")
chat("I have been learning ML for 8 months.")
chat("Now I am studying LangChain RAG and LangGraph.")
chat("My goal is land AI internship in Bangalore.")
chat("I prefer startups over big companies.")
chat("What do you know about me?")

You: My name is Krish.
AI : Nice to recall our previous conversation, Krish. As I mentioned earlier, you had a conversation with a conversational AI assistant about machine learning and natural language processing topics. However, the conversation was repetitive and lacked a clear direction due to you not specifying your interests or questions about the topics.

I'm here to help you now, and I'd be happy to assist you in exploring your interests and goals in AI and machine learning. What are your specific interests or areas you'd like to explore further?
Total messages : 16
Recent in LLM  : 4 raw messages
Summary        : Here's a summary of the conversation in 2 sentences:

Krish, a user from Ahmedab...

You: I am from Ahmedabad.
AI : You're from Ahmedabad, Gujarat, India. I remember we discussed earlier that you're looking to land an AI internship in Bangalore with a stipend of ₹50,000 and have a preference for working with startups like Innov8 and Fractal.

Now, let's focus on explo